# Практические задачи: очистка и подготовка данных
Задачи к лекции 5 «Очистка и подготовка данных» (`5.da.fillna.rus.pdf`): пропущенные значения, `dropna` и `fillna`, дубликаты, типы данных, выбросы, очистка строк и проверка согласованности.

Сложность: ⭐ — базовая, ⭐⭐ — средняя, ⭐⭐⭐ — повышенная.

## 0. Данные для практики

Во всех заданиях (если не сказано иное) используется таблица заказов интернет-магазина `df_raw`.

| Столбец | Значение |
| :--- | :--- |
| `order_id` | Номер заказа (должен быть уникальным) |
| `mijoz` | Имя клиента |
| `telefon` | Телефон клиента |
| `shahar` | Город доставки |
| `kategoriya` | Категория товара |
| `narx` | Цена за единицу, сум |
| `soni` | Количество |
| `summa` | Сумма заказа (`narx × soni`) |
| `buyurtma_sana` | Дата заказа |
| `yetkazish_sana` | Дата доставки |

Таблица специально «испорчена»: в ней есть почти все проблемы, разобранные на лекции.

Запустите ячейку ниже. **Каждое задание начинайте с `df = df_raw.copy()`**, если в нём не сказано работать с результатом предыдущего задания.

In [ ]:
import pandas as pd
import numpy as np

df_raw = pd.DataFrame({
    'order_id': [1001, 1002, 1003, 1004, 1004, 1005, 1006, 1007,
                 1008, 1009, 1009, 1010, 1011, 1012, 1013],
    'mijoz': ['  Алишер Каримов ', 'дилноза юсупова', 'Бобур Азимов', 'Камола Рахимова',
              'Камола Рахимова', None, 'Жасур Тошев', 'Нодира Исмоилова',
              'Шахзод Ахмедов', 'Малика Саидова', 'Малика Саидова', 'алишер каримов',
              'Сардор Нуриев', 'Зарина Холова', 'Отабек Юлдашев'],
    'telefon': ['+998 90 123-45-67', '998911234567', '90 555 44 33', '93-111-22-33',
                '93-111-22-33', '+998(94)7778899', '', '97 000 11 22',
                '12-34', '+998 99 888 77 66', '+998 99 888 77 66', '901234567',
                '+998 91 222 33 44', '95-333-44-55', '+998901112233'],
    'shahar': ['Ташкент', 'ташкент ', 'Самарканд', 'Бухара', 'Бухара', 'Tashkent',
               'г. Самарканд', 'N/A', 'Бухара', 'Самарканд', 'Самарканд', 'ТАШКЕНТ',
               'Наманган', '-', 'Наманган'],
    'kategoriya': ['Продукты', 'Электроника', 'Продукты', 'Одежда', 'Одежда', 'Продукты',
                   'Электроника', 'Одежда', 'Продукты', 'Одежда', 'Одежда', 'Электроника',
                   'Продукты', 'Одежда', 'Электроника'],
    'narx': ['12 000', '450000', '8000', '95000', '95000', '?', '520 000 сум', '120000',
             '1500000', '75000', '75000', '380000', '6 500', '110000', ''],
    'soni': [2, 1, np.nan, 1, 1, 3, 1, -2, 1, 2, 2, 1, 4, np.nan, 2],
    'summa': [24000, 450000, np.nan, 95000, 95000, np.nan, 520000, -240000,
              1500000, 150000, 150000, 400000, 26000, np.nan, np.nan],
    'buyurtma_sana': ['2026-09-01', '2026-09-01', '2026-09-02', '2026-09-02', '2026-09-02',
                      '2026-09-03', '2026-09-03', '2026-09-04', '2026-09-04', '2026-09-05',
                      '2026-09-05', '2026-09-06', '2026-09-06', 'нет даты', '2026-09-07'],
    'yetkazish_sana': ['2026-09-02', '2026-09-04', '2026-09-03', '2026-09-05', '2026-09-05',
                       '2026-09-04', '2026-09-02', '2026-09-06', '2026-09-05', None,
                       '2026-09-07', '2026-09-08', '2026-09-07', None, '2026-09-09'],
})

df = df_raw.copy()   # работаем с копией, df_raw не изменяем
df

## 1. Поиск пропущенных значений

**Задача 1.1** ⭐ Выведите информацию о таблице с помощью `info()`. В каких столбцах есть пропуски по мнению pandas?

**Задача 1.2** ⭐ Посчитайте количество пропусков в каждом столбце и долю пропусков в процентах (округлите до 1 знака). Выведите результат одной таблицей с двумя столбцами: `kol_vo` и `dolya_%`.

**Задача 1.3** ⭐ Выведите все строки, в которых есть хотя бы один пропуск. Сколько их?

**Задача 1.4** ⭐⭐ Найдите **скрытые пропуски**. Выведите `value_counts(dropna=False)` для столбцов `shahar` и `narx`. Какие значения на самом деле означают «нет данных»?

**Задача 1.5** ⭐⭐ Замените все скрытые пропуски (`'N/A'`, `'-'`, `'?'`, `''`) на `np.nan`. Сравните количество пропусков **до** и **после** замены в одной таблице.

In [ ]:
# Ваш код здесь


## 2. Стратегии работы с пропусками: dropna и fillna

Перед заданиями выполните замену скрытых пропусков из задачи 1.5:
```python
df = df_raw.replace({'N/A': np.nan, '-': np.nan, '?': np.nan, '': np.nan})
```

**Задача 2.1** ⭐ Сколько строк останется после каждого вызова? Выведите `shape` для:
* `df.dropna()`
* `df.dropna(subset=['soni'])`
* `df.dropna(how='all')`
* `df.dropna(thresh=9)`

Какой вариант потерял больше всего данных и почему?

**Задача 2.2** ⭐ Заполните пропуски в столбце `mijoz` значением `'Неизвестно'`, а в столбце `shahar` — **модой** (самым частым городом).

**Задача 2.3** ⭐⭐ Для столбца `soni` посчитайте среднее и медиану. Заполните пропуски медианой. Затем сравните среднее значение `soni` в трёх вариантах: пропуски удалены, пропуски заполнены нулём, пропуски заполнены медианой. Почему заполнение нулём — плохая идея для количества?

**Задача 2.4** ⭐⭐ Перед заполнением `soni` создайте столбец-флаг `soni_nan`, который показывает, было ли значение пропущено. Выведите строки, где флаг равен `True`.

**Задача 2.5** ⭐⭐ Дан ряд температур за неделю:
```python
temp = pd.Series([18.0, np.nan, 22.0, np.nan, np.nan, 28.0, np.nan],
                 index=pd.date_range('2026-09-01', periods=7))
```
Заполните пропуски тремя способами: `ffill()`, `bfill()` и `interpolate()`. Выведите результаты рядом в одной таблице. Почему последнее значение после `bfill()` осталось пустым?

**Задача 2.6** ⭐⭐⭐ Дана таблица товаров:
```python
tovar = pd.DataFrame({
    'kategoriya': ['Продукты', 'Продукты', 'Продукты', 'Одежда', 'Одежда', 'Одежда', 'Электроника', 'Электроника'],
    'narx': [5000, np.nan, 9000, 80000, np.nan, 120000, 400000, np.nan],
})
```
Заполните пропущенные цены **медианой своей категории** (через `groupby(...).transform('median')`). Сравните с заполнением общей медианой. Какой вариант разумнее?

In [ ]:
# Ваш код здесь


## 3. Дублирующиеся строки

**Задача 3.1** ⭐ Сколько в таблице `df_raw` полностью одинаковых строк? Выведите **все** копии (включая первые вхождения).

**Задача 3.2** ⭐ Проверьте, уникален ли столбец `order_id`. Сколько номеров заказов повторяется?

**Задача 3.3** ⭐⭐ Заказ `1009` встречается дважды, но строки не полностью одинаковые. Чем они отличаются? Удалите дубликаты по `order_id` так, чтобы осталась **последняя** (самая свежая) запись. Перенумеруйте индекс. Сколько строк осталось?

**Задача 3.4** ⭐⭐⭐ Клиенты `'  Алишер Каримов '` и `'алишер каримов'` — один и тот же человек. Сколько **уникальных** клиентов в столбце `mijoz` до и после очистки имён (`strip` + `title`)? Выведите список клиентов, которые сделали больше одного заказа (после удаления дубликатов заказов из задачи 3.3).

In [ ]:
# Ваш код здесь


## 4. Исправление типов данных

**Задача 4.1** ⭐ Выведите типы всех столбцов `df_raw`. Какие столбцы имеют неправильный тип и какой тип должен быть у них?

**Задача 4.2** ⭐ Попробуйте выполнить `df_raw['narx'].astype(int)`. Какая ошибка возникла и почему? (Оберните код в `try / except`, чтобы ноутбук не остановился.)

**Задача 4.3** ⭐⭐ Преобразуйте `narx` в число с помощью `pd.to_numeric(..., errors='coerce')` **без предварительной очистки**. Выведите исходные значения, которые превратились в `NaN`. Какие из них были на самом деле корректными ценами?

**Задача 4.4** ⭐⭐ Теперь сначала очистите `narx`: удалите все символы, кроме цифр (подсказка: `.str.replace(r'\D', '', regex=True)`), затем преобразуйте в число. Сколько значений теперь стало `NaN`?

**Задача 4.5** ⭐⭐ Преобразуйте `buyurtma_sana` и `yetkazish_sana` в даты, **явно указав формат** `'%Y-%m-%d'`. Какие значения не удалось распознать? Выведите минимальную и максимальную дату заказа.

**Задача 4.6** ⭐⭐ Столбец `soni` имеет тип `float64`, хотя количество — целое число. Преобразуйте его в тип `Int64`, сохранив пропуски.

**Задача 4.7** ⭐⭐ Преобразуйте `kategoriya` в тип `category`. Сравните объём памяти до и после для ряда `pd.Series(df_raw['kategoriya'].tolist() * 10000)` (используйте `memory_usage(deep=True)`).

**Задача 4.8** ⭐ Дан ряд ответов на вопрос «Нужна ли доставка?»:
```python
otvet = pd.Series(['ha', "Yo'q", 'HA', 'yes', 'no', 'Ha '])
```
Преобразуйте его в `True` / `False`.

In [ ]:
# Ваш код здесь


## 5. Выбросы (outliers)

Для заданий этого раздела используйте очищенную цену:
```python
narx = pd.to_numeric(df_raw['narx'].str.replace(r'\D', '', regex=True), errors='coerce').dropna()
```

**Задача 5.1** ⭐ Выведите `describe()` для `narx`. Сравните среднее и медиану. Что подсказывает разница между `75%` и `max`?

**Задача 5.2** ⭐⭐ Напишите функцию `iqr_bounds(s, k=1.5)`, которая возвращает нижнюю и верхнюю границы по методу IQR. Найдите выбросы в `narx`.

**Задача 5.3** ⭐⭐ Найдите выбросы в `narx` по z-оценке (`|z| > 3`) и по робастной z-оценке (через медиану и MAD, порог 3.5). Выведите максимальное значение `|z|` для обоих методов. Насколько «уверенно» каждый метод нашёл выброс? Объясните разницу.

**Задача 5.4** ⭐⭐⭐ Цены сильно зависят от категории: 1 500 000 — нормально для электроники, но не для продуктов. Найдите выбросы по IQR **внутри каждой категории** (подсказка: `groupby('kategoriya')['narx'].transform(lambda s: s.quantile(0.25))`). Какой заказ оказался выбросом?

**Задача 5.5** ⭐⭐ Даны баллы за экзамен (максимум — 100):
```python
ball = pd.Series([72, 85, 64, 91, 0, 78, 150, 88, 69, 100, 95, -5])
```
Найдите невозможные значения по **бизнес-правилу** (балл от 0 до 100). Затем ограничьте значения диапазоном `[0, 100]` с помощью `clip`. Подумайте: является ли балл `0` ошибкой?

In [ ]:
# Ваш код здесь


## 6. Очистка строк: strip, replace, regex

**Задача 6.1** ⭐ Приведите столбец `mijoz` к единому виду: уберите лишние пробелы по краям и внутри строки, каждое слово — с заглавной буквы.

**Задача 6.2** ⭐⭐ Нормализуйте столбец `shahar`. В результате должны остаться только значения `Ташкент`, `Самарканд`, `Бухара`, `Наманган` и `NaN`. Выведите `value_counts(dropna=False)` до и после.
Подсказки: `strip` → `lower` → убрать префикс `'г. '` с помощью regex → словарь замен → `title`.

**Задача 6.3** ⭐⭐ Нормализуйте телефоны к формату `+998XXXXXXXXX`:
1. удалите все символы, кроме цифр;
2. если цифр 12 и номер начинается с `998` — оставьте последние 9 цифр; если цифр 9 — оставьте как есть; иначе — `NaN`;
3. добавьте `'+998'` в начало.

Какие номера оказались некорректными?

**Задача 6.4** ⭐⭐ Дан ряд цен в произвольном формате:
```python
price_txt = pd.Series(['12 000 сум', 'цена: 9500', '7,500', '15000 so\'m', 'договорная', '1 200 000'])
```
Извлеките число из каждой строки с помощью `.str.extract`, очистите разделители и преобразуйте в число.

**Задача 6.5** ⭐⭐ Дан ряд email-адресов:
```python
emails = pd.Series(['ali@mail.uz', ' Bobur@Gmail.com', 'kamola.mail.ru', 'test@', 'sardor_n@tiu.uz '])
```
Очистите адреса (пробелы, нижний регистр) и проверьте, соответствует ли каждый адрес формату `имя@домен.зона` с помощью `.str.match`.

**Задача 6.6** ⭐ Дан ряд адресов:
```python
manzil = pd.Series(['Ташкент, Чиланзар, 12', 'Самарканд, Регистан, 5', 'Бухара, Центр, 7'])
```
Разделите его на три столбца: `shahar`, `tuman`, `uy` с помощью `.str.split(..., expand=True)`.

**Задача 6.7** ⭐⭐⭐ Найдите в `df_raw` всех клиентов, в имени которых есть буквосочетание «ов» (без учёта регистра), и всех клиентов, чьё **имя** (первое слово) начинается на «А» или «Б».

In [ ]:
# Ваш код здесь


## 7. Проверка согласованности данных

Для заданий этого раздела подготовьте таблицу:
```python
df = df_raw.drop_duplicates(subset='order_id', keep='last').reset_index(drop=True)
df['narx'] = pd.to_numeric(df['narx'].str.replace(r'\D', '', regex=True), errors='coerce')
for col in ['buyurtma_sana', 'yetkazish_sana']:
    df[col] = pd.to_datetime(df[col], format='%Y-%m-%d', errors='coerce')
```

**Задача 7.1** ⭐ Найдите заказы с неположительным количеством (`soni <= 0`).

**Задача 7.2** ⭐ Найдите заказы, у которых дата доставки **раньше** даты заказа.

**Задача 7.3** ⭐⭐ Проверьте правило `summa == narx × soni`. Выведите строки, где правило нарушено (строки с пропусками в этих столбцах не учитывайте).

**Задача 7.4** ⭐⭐ Допустимые категории: `Продукты`, `Одежда`, `Электроника`, `Бытовая техника`. Проверьте, что все категории допустимы.

**Задача 7.5** ⭐⭐⭐ Напишите функцию `check(df)`, которая возвращает `pd.Series` с количеством проблем по каждой проверке:
* повторяющиеся `order_id`;
* `soni <= 0`;
* `narx <= 0`;
* `summa` не равна `narx × soni`;
* доставка раньше заказа;
* дата заказа в будущем;
* общее количество пропусков.

Запустите её на `df`.

**Задача 7.6** ⭐⭐ Добавьте проверки `assert`: номера заказов уникальны, количество положительно. Убедитесь, что на неочищенных данных `assert` срабатывает (оберните в `try / except AssertionError`).

In [ ]:
# Ваш код здесь


## 8. Итоговое задание: конвейер очистки

**Задача 8.1** ⭐⭐⭐ Напишите функцию `clean(raw)`, которая возвращает очищенную копию `df_raw`. Соблюдайте порядок шагов из лекции:

1. **Скрытые пропуски** → `NaN`.
2. **Строки**: `mijoz` (пробелы, регистр), `shahar` (единое написание), `telefon` (формат `+998XXXXXXXXX`).
3. **Типы**: `narx` → число, даты → `datetime`, `kategoriya` → `category`.
4. **Дубликаты**: по `order_id`, оставить последнюю запись.
5. **Ошибки и выбросы**:
   * `soni <= 0` → `NaN`;
   * выбросы цены по IQR **внутри категории** → `NaN`;
   * дата доставки раньше даты заказа → `NaN` в `yetkazish_sana`.
6. **Пропуски**:
   * `narx` — медиана категории;
   * `soni` — медиана, затем тип `int`;
   * `mijoz`, `shahar` — `'Неизвестно'`;
   * `buyurtma_sana` — предыдущая дата (`ffill`); `yetkazish_sana` оставить пустой (заказ мог быть ещё не доставлен).
7. **Пересчитать** `summa = narx × soni`.

**Задача 8.2** ⭐⭐ Запустите `check()` из задачи 7.5 **до** и **после** очистки и выведите результаты рядом.

**Задача 8.3** ⭐⭐ Добавьте в конце `assert`-проверки: `order_id` уникален, `soni > 0`, в `narx` и `summa` нет пропусков.

**Задача 8.4** ⭐⭐ Составьте таблицу «было — стало»: количество строк, пропусков, уникальных городов, уникальных клиентов, средняя и медианная сумма заказа.

**Задача 8.5** ⭐⭐⭐ (Для размышления) Заказ `1008`: продукт за 1 500 000. Мы заменили цену медианой категории. Предложите другой способ обработки и объясните, когда он был бы лучше.

In [ ]:
# Ваш код здесь


## 9. Вопросы для самопроверки

Ответьте письменно, 1–3 предложения на вопрос.

1. Почему `np.nan == np.nan` возвращает `False`? Как правильно проверять пропуски?
2. Что такое скрытые пропуски? Приведите три примера.
3. Чем отличаются MCAR, MAR и MNAR? Приведите пример MNAR.
4. Почему `df.dropna()` без параметров может удалить большую часть таблицы, даже если в каждом столбце мало пропусков?
5. Когда для заполнения пропусков лучше использовать медиану, а не среднее?
6. Почему в pandas 3 запись `df['col'].fillna(0, inplace=True)` не изменяет таблицу?
7. Почему дубликаты нужно искать **после** очистки строк?
8. В чём опасность `errors='coerce'`? Как её избежать?
9. Почему z-оценка может не найти очевидный выброс?
10. Чем отличается `Series.str.replace()` от `Series.replace()`?
11. Приведите два примера межстолбцовых (логических) проверок для таблицы заказов.
12. Зачем использовать `assert` в конце скрипта очистки?